In [ ]:

import matplotlib.pyplot as plt
import numpy as np
import pprint
import joblib
from skimage.feature import hog
from skimage.feature import local_binary_pattern
import cv2
import os

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler

IMG_SIZE = (128,128)
TEST_SIZE = 0.2


# =========================================================
# CELL : EXTRACT ZIP DATASET
# =========================================================

import os
import zipfile

# ZIP file path
ZIP_FILE = "archive.zip"

# Folder where dataset will be extracted
DATASET_FOLDER = "dataset"

# Create folder
os.makedirs(DATASET_FOLDER, exist_ok=True)

# Extract ZIP
with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
    zip_ref.extractall(DATASET_FOLDER)

print("ZIP Extracted Successfully")


# Dataset path for further use
dataset_path = "dataset/data"

print("Dataset Path :", dataset_path)



# =========================================================
# PRINT CLASSES NAME
# =========================================================

classes = os.listdir(dataset_path)

print("\nClasses Found :")

for class_name in classes:
    print(class_name)


def extract_features(img_path):
    img = cv2.imread(img_path)
    if img is None:
        return None

    img = cv2.resize(img, IMG_SIZE)

    # Corrected Color Conversion
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    feats = []

    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    for ch in cv2.split(hsv):
        feats += [ch.mean(), ch.std()]

    # LBP Features
    lbp = local_binary_pattern(gray, P=8, R=1, method="uniform")
    hist, _ = np.histogram(lbp, bins=10, range=(0,10), density=True)
    feats += list(hist)

    # Edge Features
    edges = cv2.Canny(gray, 100, 200)
    feats.append(edges.mean())

    # Hu Moments
    moments = cv2.moments(gray)
    hu = cv2.HuMoments(moments).flatten()
    feats += list(-np.sign(hu) * np.log10(np.abs(hu) + 1e-10))

    # Contours
    cnts, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if cnts:
        c = max(cnts, key=cv2.contourArea)
        feats += [cv2.contourArea(c), cv2.arcLength(c, True)]
    else:
        feats += [0,0]

    # HOG Features
    hog_feats = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8,8),
        cells_per_block=(2,2),
        feature_vector=True
    )

    feats += list(hog_feats)

    return np.array(feats, dtype=np.float32)


# =========================================================
# LOAD DATASET
# =========================================================

X, y = [], []

with_mask = "dataset/data/with_mask"

for file in os.listdir(with_mask):

    path = os.path.join(with_mask, file)

    f = extract_features(path)

    if f is not None:
        X.append(f)
        y.append("withmask")
    else:
        print(f"Skip Corrupted : {file}")


without_mask = "dataset/data/without_mask"

for file in os.listdir(without_mask):

    path = os.path.join(without_mask, file)

    f = extract_features(path)

    if f is not None:
        X.append(f)
        y.append("withoutmask")
    else:
        print(f"Skip Corrupted : {file}")


X = np.array(X)
y = np.array(y)


# =========================================================
# TRAIN TEST SPLIT
# =========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=42,
    stratify=y
)


# =========================================================
# FEATURE SCALING
# =========================================================

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# =========================================================
# RANDOM FOREST MODEL
# =========================================================

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

print("\n===== RANDOM FOREST RESULT =====\n")

print(classification_report(
    y_test,
    y_pred,
    target_names=["withmask", "withoutmask"]
))

accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy : {accuracy * 100:.2f}%")


# =========================================================
# LINEAR SVM MODEL
# =========================================================

from sklearn.svm import LinearSVC

svm_model = LinearSVC(
    C=1.0,
    max_iter=2000
)

svm_model.fit(X_train, y_train)

y_pred = svm_model.predict(X_test)

print("\n===== LINEAR SVC RESULT =====\n")

print(classification_report(
    y_test,
    y_pred,
    target_names=["withmask", "withoutmask"]
))

accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy : {accuracy * 100:.2f}%")


# =========================================================
# SAVE MODEL
# =========================================================

joblib.dump(svm_model, "mask_classifier.pkl")
joblib.dump(scaler, "scaler.pkl")

print("\nModel Saved Successfully")


In [ ]:
# =========================================================
# UNIVERSAL BINARY IMAGE CLASSIFICATION CODE
# Only Change Dataset Path
# Everything Else Automatic
# =========================================================

import os
import cv2
import numpy as np

from skimage.feature import hog

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report


# =========================================================
# ONLY CHANGE THIS PATH
# =========================================================

DATASET_PATH = "dataset"

# Example Dataset Structure:
#
# dataset/
# ├── cat/
# └── dog/
#
# OR
#
# dataset/
# ├── with_mask/
# └── without_mask/
#
# OR
#
# dataset/
# ├── real/
# └── fake/
#
# Any 2 folders = Binary Classification


# =========================================================
# SETTINGS
# =========================================================

IMG_SIZE = (64, 64)
TEST_SIZE = 0.2


# =========================================================
# GET CLASSES AUTOMATICALLY
# =========================================================

classes = os.listdir(DATASET_PATH)

print("\nClasses Found :")

for c in classes:
    print(c)

# Binary Check
if len(classes) != 2:
    print("\nERROR : Dataset Must Contain Exactly 2 Classes")
    exit()


# =========================================================
# FEATURE EXTRACTION FUNCTION
# =========================================================

def extract_features(img_path):

    # Read Image
    img = cv2.imread(img_path)

    # Skip Corrupted Images
    if img is None:
        return None

    # Resize
    img = cv2.resize(img, IMG_SIZE)

    # Convert to Grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # HOG Features
    features = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8,8),
        cells_per_block=(2,2),
        feature_vector=True
    )

    return features


# =========================================================
# LOAD DATASET AUTOMATICALLY
# =========================================================

X = []
y = []

for class_name in classes:

    class_path = os.path.join(DATASET_PATH, class_name)

    print(f"\nLoading : {class_name}")

    for file in os.listdir(class_path):

        img_path = os.path.join(class_path, file)

        features = extract_features(img_path)

        if features is not None:

            X.append(features)
            y.append(class_name)

        else:
            print("Skipped :", file)


# Convert to NumPy Arrays
X = np.array(X)
y = np.array(y)

print("\nTotal Images :", len(X))


# =========================================================
# TRAIN TEST SPLIT
# =========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=42,
    stratify=y
)


# =========================================================
# FEATURE SCALING
# =========================================================

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# =========================================================
# MODEL TRAINING
# =========================================================

model = LinearSVC()

model.fit(X_train, y_train)

print("\nModel Trained Successfully")


# =========================================================
# PREDICTION
# =========================================================

y_pred = model.predict(X_test)


# =========================================================
# EVALUATION
# =========================================================

accuracy = accuracy_score(y_test, y_pred)

print("\nAccuracy :", accuracy * 100)


print("\nClassification Report :\n")

print(classification_report(y_test, y_pred))


# =========================================================
# TEST SINGLE IMAGE
# =========================================================

sample_class = classes[0]

sample_path = os.path.join(
    DATASET_PATH,
    sample_class,
    os.listdir(os.path.join(DATASET_PATH, sample_class))[0]
)

features = extract_features(sample_path)

features = scaler.transform([features])

prediction = model.predict(features)

print("\nSample Prediction :", prediction[0])